# 02 — YOLO Training (5-Fold Cross-Validation)

Trains YOLO26 on each of the 5 folds produced by `01_prepare_dataset.ipynb`,
then retrains a **final model on all 80% training data** (no held-out val fold).

The final model weights (`runs/yolo26/final/weights/best.pt`) are consumed by:
- `04_tracker_architecture_botsort.ipynb`
- `04b_tracker_architecture_bytetrack.ipynb`
- `05_tracker_hyperparameter_tuning_botsort.ipynb`
- `07_realtime_demo.ipynb`
- `08_longform_demo.ipynb`

Prerequisites: `01_prepare_dataset.ipynb`


## 0 — Configuration

In [14]:
from pathlib import Path
from ultralytics import YOLO
import torch, yaml, json

REPO_ROOT  = Path("../")
YOLO_DIR   = REPO_ROOT / "data/yolo_dataset_cv"
RUNS_DIR   = (REPO_ROOT / "runs" / "yolo26").resolve()
N_FOLDS    = 5

MODEL_SIZE = "yolo26n"   # n / s / m / l / x
EPOCHS     = 10
IMGSZ      = 320
BATCH      = 32

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

with open(REPO_ROOT / "data/splits/splits.json") as f:
    splits = json.load(f)
CLASSES = splits["classes"]
print(f"Classes: {CLASSES}")


Device: mps
Classes: ['Boost', 'Charge', 'Defense', 'Glide', 'HP', 'Offense', 'Top Speed', 'Turn', 'Weight']


## 0b — Diagnostics Helpers

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

_COLORS = plt.cm.tab20.colors

def plot_all_folds_diagnostics(runs_dir, n_folds, final=True):
    """Plot train AND val curves for all folds + optional final model.

    Layout: 2 rows x 3 cols
      Row 0 — Train : Box Loss  |  mAP@50 (empty, val-only)  |  mAP@50:95 (empty)
      Row 1 — Val   : Box Loss  |  mAP@50                    |  mAP@50:95
    Train curves are solid lines; the Final model is plotted in black.
    """
    runs_dir = Path(runs_dir)
    fold_colors = plt.cm.tab10.colors

    # (title, train_key_fn, val_key_fn, ylabel)
    col_defs = [
        (
            'Box Loss',
            lambda df: next((c for c in df.columns if 'box_loss' in c and 'train' in c), None),
            lambda df: next((c for c in df.columns if 'box_loss' in c and 'val'   in c), None),
            'Box Loss',
        ),
        (
            'mAP@50',
            lambda df: None,   # val-only metric
            lambda df: next((c for c in df.columns if 'mAP50' in c and '95' not in c), None),
            'mAP@50',
        ),
        (
            'mAP@50:95',
            lambda df: None,
            lambda df: next((c for c in df.columns if 'mAP50-95' in c), None),
            'mAP@50:95',
        ),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharex=False)
    fig.suptitle('YOLO26 CV Folds — Train / Val Curves', fontweight='bold')
    for row, rlabel in enumerate(['Train', 'Val']):
        for col, (title, _, __, ylabel) in enumerate(col_defs):
            ax = axes[row][col]
            ax.set_title(f'{title} ({rlabel})')
            ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
            ax.grid(True, alpha=0.3)

    key_fns = [(d[1], d[2]) for d in col_defs]

    def _plot_run(df, color, lw, label_prefix):
        x = df['epoch'] if 'epoch' in df.columns else range(len(df))
        for col, (train_fn, val_fn) in enumerate(key_fns):
            col_name = train_fn(df)
            if col_name:
                axes[0][col].plot(x, df[col_name], color=color, lw=lw, ls='-', label=label_prefix)
            col_name = val_fn(df)
            if col_name:
                axes[1][col].plot(x, df[col_name], color=color, lw=lw, ls='--', label=label_prefix)

    for fold in range(n_folds):
        csv_path = runs_dir / f"fold_{fold}" / "results.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path); df.columns = df.columns.str.strip()
        _plot_run(df, fold_colors[fold % len(fold_colors)], 2, f'Fold {fold}')

    if final:
        csv_final = runs_dir / "final" / "results.csv"
        if csv_final.exists():
            df = pd.read_csv(csv_final); df.columns = df.columns.str.strip()
            _plot_run(df, 'black', 2.5, 'Final')

    for row in axes:
        for ax in row:
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                ax.legend(fontsize=8)

    plt.tight_layout()
    out = runs_dir / "fold_training_curves.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved: {out}")


def plot_per_class_ap(run_dir, fold_label, data_yaml, model_path):
    """Per-class AP@50 bar chart for a single model."""
    if not Path(model_path).exists():
        print(f"  [{fold_label}] weights not found — skipping per-class AP")
        return None
    m = YOLO(str(model_path)).val(data=str(Path(data_yaml).resolve()),
                                   imgsz=IMGSZ, device=DEVICE, verbose=False)
    ap_idx  = m.box.ap_class_index.tolist() if hasattr(m.box.ap_class_index, 'tolist') else list(m.box.ap_class_index)
    ap_vals = m.box.ap50.tolist()           if hasattr(m.box.ap50,           'tolist') else list(m.box.ap50)
    class_labels = [CLASSES[i] for i in ap_idx]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(class_labels, ap_vals,
                  color=[_COLORS[i % len(_COLORS)] for i in range(len(class_labels))],
                  edgecolor='white')
    for bar, val in zip(bars, ap_vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', fontsize=8, fontweight='bold')
    ax.axhline(y=m.box.map50, color='red', ls='--', lw=1.5,
               label=f'mAP@50={m.box.map50:.3f}')
    ax.set_ylabel('AP@50'); ax.set_title(f'{fold_label} — Per-Class AP@50', fontweight='bold')
    ax.set_ylim(0, 1.1); ax.legend(); ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=30, ha='right'); plt.tight_layout()
    out = Path(run_dir) / "diagnostics_per_class_ap.png"
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  mAP@50={m.box.map50:.4f}  mAP@50:95={m.box.map:.4f}  Saved: {out}")
    return m


def plot_sample_predictions(model_path, val_img_dir, title='Final Model', n=9):
    """Show a 3×3 grid of random val images with predicted bounding boxes.
    val_img_dir may be a directory of images OR a .txt manifest file.
    """
    val_img_dir = Path(val_img_dir)
    if val_img_dir.suffix == '.txt':
        imgs = [Path(p) for p in val_img_dir.read_text().splitlines() if p.strip()]
        imgs = sorted([p for p in imgs if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    else:
        imgs = sorted([p for p in val_img_dir.iterdir()
                       if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
    if not imgs:
        print(f"  No images found in {val_img_dir}")
        return

    model = YOLO(str(model_path))
    np.random.seed(42)
    chosen = [imgs[i] for i in np.random.choice(len(imgs), min(n, len(imgs)), replace=False)]

    cols = 3
    rows_n = (len(chosen) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols*5, rows_n*4))
    axes = np.array(axes).flatten()
    fig.suptitle(f'{title} — Sample Predictions', fontweight='bold')

    for ax, img_path in zip(axes, chosen):
        res = model.predict(str(img_path), conf=0.25, verbose=False)
        ax.imshow(res[0].plot()[:, :, ::-1])
        ax.set_title(f'{len(res[0].boxes)} detections', fontsize=9)
        ax.axis('off')
    for ax in axes[len(chosen):]:
        ax.axis('off')

    plt.tight_layout()
    out = Path(model_path).parent.parent / "sample_predictions.png"
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {out}")

print("Diagnostics helpers ready.")


## 1 — 5-Fold Training

In [17]:
fold_results = {}

for fold in range(N_FOLDS):
    data_yaml = YOLO_DIR / f"fold_{fold}" / "data.yaml"
    if not data_yaml.exists():
        print(f"Fold {fold}: data.yaml missing — run 01_prepare_dataset first")
        continue

    print(f"\n{'='*50}")
    print(f"Fold {fold}/5  ({MODEL_SIZE}, {EPOCHS} epochs)")
    print(f"{'='*50}")

    model = YOLO(f"{MODEL_SIZE}.pt")
    results = model.train(
        data    = str(data_yaml.resolve()),
        epochs  = EPOCHS,
        imgsz   = IMGSZ,
        batch   = BATCH,
        device  = DEVICE,
        project = str(RUNS_DIR),
        name    = f"fold_{fold}",
        exist_ok= True,
        plots   = True,
        val     = True,
        agnostic_nms=True,
        nms=True,
        max_det=20,
        workers=0,
        cache=True,
        freeze=10
    )
    fold_results[fold] = {
        "map50":    results.results_dict.get("metrics/mAP50(B)", 0),
        "map50_95": results.results_dict.get("metrics/mAP50-95(B)", 0),
        "weights":  str(RUNS_DIR / f"fold_{fold}" / "weights" / "best.pt"),
    }
    print(f"Fold {fold} mAP@0.5: {fold_results[fold]['map50']:.4f}")

print("\nFold summary:")
maps = [v["map50"] for v in fold_results.values()]
print(f"  mAP@0.5  mean={np.mean(maps):.4f}  std={np.std(maps):.4f}  "
      f"min={min(maps):.4f}  max={max(maps):.4f}")

# ── All-folds combined training curves (plotted once, after all folds) ────
plot_all_folds_diagnostics(RUNS_DIR, N_FOLDS, final=False)



Fold 0/5  (yolo11n, 10 epochs)
New https://pypi.org/project/ultralytics/8.4.42 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.40 🚀 Python-3.12.8 torch-2.11.0 MPS (Apple M4 Pro)
engine/trainer: agnostic_nms=True, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/joh11678/.ws/KARTector/data/yolo_dataset_cv_reduced/fold_0/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=20, mixup=0.0, mode=train, model=yolo11n.

## 2 — Final Model (all 80% data)

In [ ]:
full_yaml = YOLO_DIR / "full_train" / "data.yaml"
print(f"Training final model on: {full_yaml}")

final_model = YOLO(f"{MODEL_SIZE}.pt")
final_results = final_model.train(
    data    = str(full_yaml.resolve()),
    epochs  = EPOCHS,
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = DEVICE,
    project = str(RUNS_DIR),
    name    = "final",
    exist_ok= True,
    plots   = True,
    val     = True,
    agnostic_nms=True,
    nms=True,
    max_det=20,
    workers=0,
    cache=True,
    freeze=10
)
print(f"Final model mAP@0.5: {final_results.results_dict.get('metrics/mAP50(B)',0):.4f}")
final_weights = RUNS_DIR / "final" / "weights" / "best.pt"
print(f"Weights: {final_weights}")

# ── Combined curves: all folds + final model overlaid ─────────────────────
plot_all_folds_diagnostics(RUNS_DIR, N_FOLDS, final=True)

# ── Final model per-class AP ──────────────────────────────────────────────
plot_per_class_ap(
    run_dir    = RUNS_DIR / "final",
    fold_label = "Final Model",
    data_yaml  = full_yaml,
    model_path = str(final_weights),
)

# ── 3×3 sample predictions (final model only) ─────────────────────────────
val_img_dir = YOLO_DIR / "full_train" / "val.txt"
if not val_img_dir.exists():
    val_img_dir = YOLO_DIR / "fold_0" / "val.txt"
plot_sample_predictions(
    model_path  = str(final_weights),
    val_img_dir = val_img_dir,
    title       = "Final Model",
    n           = 9,
)
